# **Residual stream: практика**

В уроках «Residual stream как канал связи», «MLP как key-value память» и «Композиция голов и virtual heads» все считалось на маленьких векторах руками. Здесь те же три инструмента применяются к GPT-2 small из TransformerLens: 12 слоев, по 12 голов внимания, residual stream размерности 768.

В этой практике вы:

* разложите логит токена « Paris» на промпте «The Eiffel Tower is in the city of» на прямые вклады блоков и голов (direct logit attribution);
* найдете нейрон MLP, который сильнее всех напрямую продвигает « Paris», и посмотрите, какие токены продвигает его вектор-значение;
* посчитаете по весам оценки Q-, K- и V-композиции между головами слоев 4 и 5 ([Elhage et al., 2021](https://transformer-circuits.pub/2021/framework/index.html)).

Задания пронумерованы, ответы на них вводятся в уроке «Residual stream: Практика».

## **Подготовка окружения**
Хватит CPU: модель небольшая, и все вычисления здесь — прямые проходы.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)  # здесь только прямые проходы
model = HookedTransformer.from_pretrained("gpt2-small", device="cpu")
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_model, model.cfg.d_mlp)

## **Промпт и предсказание**

Прогоним промпт и сохраним кэш всех активаций. Логит токена — его координата в выходе модели на последней позиции.

In [ ]:
prompt = "The Eiffel Tower is in the city of"
tokens = model.to_tokens(prompt)
print(model.to_str_tokens(prompt))
logits, cache = model.run_with_cache(tokens)
paris = model.to_single_token(" Paris")

top = logits[0, -1].topk(5)
for value, index in zip(top.values, top.indices):
    print(repr(model.to_string(index)), round(value.item(), 2))
print("logit Paris:", round(logits[0, -1, paris].item(), 2))

**Задание 1. Какой токен модель ставит на первое место после этого промпта?**

**Задание 2. Чему равен логит токена « Paris»? Ответ округлите до сотых.**

## **Разложение логита по блокам**

Финальное состояние потока — сумма эмбеддинга токена, позиционного эмбеддинга и вкладов всех блоков внимания и MLP:

$$h_L = h_0 + \sum_{l} (a_l + m_l).$$

Перед unembedding стоит финальный LayerNorm. Если зафиксировать его масштаб на значении из этого прогона, он действует на каждое слагаемое одинаково, и логит раскладывается точно: `cache.decompose_resid(..., apply_ln=True)` возвращает слагаемые после LayerNorm, и прямой вклад слагаемого — его скалярное произведение со столбцом $W_U$ токена « Paris». К сумме вкладов добавляется свободный член $b_U$.

In [ ]:
stack, labels = cache.decompose_resid(layer=-1, pos_slice=-1, apply_ln=True, return_labels=True)
w = model.W_U[:, paris]  # столбец unembedding токена « Paris»
contrib = ...  # Ваш код здесь

print("sum + b_U:", round((contrib.sum() + model.b_U[paris]).item(), 3), " logit:", round(logits[0, -1, paris].item(), 3))
for label, value in sorted(zip(labels, contrib.tolist()), key=lambda x: -x[1]):
    print(f"{label:>12} {value:7.3f}")

**Задание 3. Какой блок дает наибольший прямой вклад в логит « Paris»?**

**Задание 4. Чему равен этот вклад? Ответ округлите до сотых.**

## **Вклады голов внимания**

Вклад блока внимания — сумма вкладов его голов. `cache.stack_head_results(..., apply_ln=True)` возвращает выход каждой головы после $W_O$ с тем же зафиксированным масштабом LayerNorm.

In [ ]:
head_stack, head_labels = cache.stack_head_results(layer=-1, pos_slice=-1, apply_ln=True, return_labels=True)
head_contrib = ...  # Ваш код здесь

for label, value in sorted(zip(head_labels, head_contrib.tolist()), key=lambda x: -x[1])[:5]:
    print(f"{label:>6} {value:7.3f}")

**Задание 5. Какая голова дает наибольший прямой вклад в логит « Paris»?** Голову записывайте как «слой.голова», например 1.2.

**Задание 6. Чему равен вклад этой головы? Ответ округлите до сотых.**

## **MLP как key-value память**

Выход MLP-слоя $l$ — сумма векторов-значений с весами-активациями:

$$m_l = \sum_i \sigma(k_i^\top h)\, v_i + b_{out},$$

где $\sigma(k_i^\top h)$ — активация нейрона $i$ (в TransformerLens — `cache["post", l]`), а $v_i$ — строка `model.W_out[l]` с номером $i$. Поэтому прямой вклад нейрона $i$ в логит — его активация, умноженная на скалярное произведение $\langle v_i, w \rangle$ и деленная на масштаб $s$ финального LayerNorm (`cache["ln_final.hook_scale"]`). Возьмем слой 9.

In [ ]:
layer = 9
act = cache["post", layer][0, -1]
s = cache["ln_final.hook_scale"][0, -1, 0]
neuron_contrib = ...  # Ваш код здесь

block = labels.index(f"{layer}_mlp_out")
print("нейроны + b_out:", round((neuron_contrib.sum() + model.b_out[layer] @ w / s).item(), 3), " блок:", round(contrib[block].item(), 3))
top = neuron_contrib.topk(5)
for value, index in zip(top.values, top.indices):
    print(f"neuron {index.item():>4}  activation {act[index].item():6.3f}  contribution {value.item():6.3f}")

**Задание 7. Какой нейрон слоя 9 дает наибольший прямой вклад в логит « Paris»? Запишите его номер.**

**Задание 8. Чему равна активация этого нейрона? Ответ округлите до сотых.**

**Задание 9. Чему равен прямой вклад этого нейрона? Ответ округлите до трех знаков после точки.**

Вектор-значение нейрона можно спроецировать на весь словарь сразу: так видно, какие токены этот нейрон продвигает, когда срабатывает, на любом входе.

In [ ]:
neuron = ...  # Ваш код здесь
promoted = (model.W_out[layer, neuron] @ model.W_U).topk(10).indices
print([model.to_string(t) for t in promoted])

**Задание 10. Какие токены продвигает вектор-значение этого нейрона?**

## **Композиция по весам**

Elhage et al. меряют, насколько голова позднего слоя читает выход головы раннего слоя, нормой Фробениуса произведения матриц: для K-композиции $\|W_{QK}^{h_2}W_{OV}^{h_1}\|_F / (\|W_{QK}^{h_2}\|_F\,\|W_{OV}^{h_1}\|_F)$, для Q- и V-композиции — так же с $W_{QK}^{h_2\,T}$ и $W_{OV}^{h_2}$. В TransformerLens это `model.all_composition_scores(mode)`, тензор `[слой 1, голова 1, слой 2, голова 2]`. Данные не нужны: оценка считается по весам. В статье Elhage et al. из этой оценки вычитают ее среднее для случайных матриц той же формы; `all_composition_scores` возвращает оценку без этой поправки, поэтому ее сравнивают с другими оценками той же модели. Найдем пару голов слоев 4 и 5 с наибольшей оценкой K-композиции и сравним ее со средней по всем 144 парам этих слоев.

In [ ]:
scores = {mode: model.all_composition_scores(mode)[4, :, 5, :] for mode in "QKV"}  # [голова слоя 4, голова слоя 5]
h1, h2 = ...  # Ваш код здесь

print(f"pair 4.{h1} -> 5.{h2}")
for mode in "QKV":
    print(mode, round(scores[mode][h1, h2].item(), 4))
print("mean K over 144 pairs:", round(scores["K"].mean().item(), 4))

**Задание 11. Между какими головами слоев 4 и 5 оценка K-композиции наибольшая?**

**Задание 12. Чему равна эта оценка K-композиции? Ответ округлите до трех знаков.**

**Задание 13. Чему равна средняя оценка K-композиции по всем 144 парам голов слоев 4 и 5? Ответ округлите до четырех знаков после точки.**

## **Заключение**

- Логит раскладывается на прямые вклады блоков точно, если зафиксировать масштаб финального LayerNorm: сумма вкладов плюс $b_U$ совпадает с логитом модели.
- Вклад блока внимания раскладывается дальше по головам, вклад MLP-слоя — по нейронам: активация нейрона умножается на проекцию его вектора-значения на столбец unembedding.
- Вектор-значение нейрона, спроецированный на словарь, показывает, какие токены нейрон продвигает на любом входе, — это и есть чтение MLP как key-value памяти.
- Оценки композиции считаются по весам, без данных, и показывают, какие пары голов могут читать выходы друг друга. Что пара действительно работает вместе на тексте, проверяют вмешательством: это темы уроков о схемах модуля «Mechanistic Interpretability».